# Flash-Attention GPT-2 — Colab Pro runner

Custom CUDA flash-attention kernels vs PyTorch SDPA for GPT-2 inference.

**Before running:** set the runtime to GPU via *Runtime → Change runtime type → GPU*.

In [ ]:
# 0) Confirm a GPU is attached
!nvidia-smi

In [ ]:
# 1) Clone the repo (or `git pull` if it already exists)
import os
REPO = 'https://github.com/sudhirpol522/Flash-Attention.git'
if not os.path.isdir('Flash-Attention'):
    !git clone $REPO
%cd Flash-Attention
!git pull --ff-only || true

In [ ]:
# 2) Install dependencies (torch is usually preinstalled in Colab)
!pip install -q -r requirements.txt

In [ ]:
# 3) Correctness + throughput. RUN THIS FIRST and confirm RESULT: PASS.
# The first cuda call JIT-compiles kvcache_kernels.cu (~1-2 min).
!python benchmark.py --model gpt2 --batch 8 --prompt_len 256 --new_tokens 128

In [ ]:
# 4) Forward-pass profiling: writes results.csv and plots/*.png
!python profile_forward.py --model gpt2 --batch 8 --prompt_lens 128 256 512

In [ ]:
# 5) Show the generated plots inline
from IPython.display import Image, display
for name in ['latency_prefill.png', 'latency_decode.png', 'speedup.png']:
    p = f'plots/{name}'
    if os.path.exists(p):
        display(Image(filename=p))

In [ ]:
# 6) Inspect the raw numbers
import pandas as pd
pd.read_csv('results.csv')